In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("=== BAB 10: DATA ANALYTIC (MODELING & IMPUTASI) ===")

# --- A. PELATIHAN MODEL ---
df_train = pd.read_csv('../data/processed/volve_train_clean.csv')
feature_cols = ['BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'AVG_DOWNHOLE_PRESSURE', 'AVG_WELLHEAD_PRESSURE', 'AVG_TEMPERATURE']

X = df_train[feature_cols]
y = df_train['AVG_CHOKE_SIZE_P']

# Split & Transformasi (Mengulang proses Bab 9 secara ringkas agar variabel terbaca)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = joblib.load('../models/scaler.pkl')
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Inisialisasi dan Training
lr_model = LinearRegression().fit(X_train_scaled, y_train)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)
xgb_model = XGBRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)

# --- B. EVALUASI MODEL ---
def evaluate(y_true, y_pred, name):
    print(f"--- {name} ---")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"MAE  : {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"R2   : {r2_score(y_true, y_pred):.4f}\n")

evaluate(y_test, lr_model.predict(X_test_scaled), "Linear Regression")
evaluate(y_test, rf_model.predict(X_test_scaled), "Random Forest")
evaluate(y_test, xgb_model.predict(X_test_scaled), "XGBoost")

# --- C. FEATURE IMPORTANCE ----
importances = rf_model.feature_importances_ * 100
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=np.array(feature_cols)[indices], palette="viridis")
plt.title("Feature Importance - Random Forest", fontweight='bold')
plt.xlabel("Tingkat Kepentingan (%)")
plt.show()

# Simpan Model Terbaik
joblib.dump(rf_model, '../models/best_rf_model.pkl')

# --- D. EKSEKUSI IMPUTASI (TARGET AKHIR) ---
print("=== MISI IMPUTASI DATA ===")
df_impute = pd.read_csv('../data/processed/volve_impute_clean.csv')

# Normalisasi data yang akan diimputasi
X_missing_scaled = scaler.transform(df_impute[feature_cols])

# Prediksi bukaan choke
df_impute['AVG_CHOKE_SIZE_P'] = np.round(rf_model.predict(X_missing_scaled), 2)

# Gabung dan Urutkan
df_final = pd.concat([df_train, df_impute], ignore_index=True)
df_final['DATEPRD'] = pd.to_datetime(df_final['DATEPRD'])
df_final = df_final.sort_values(by=['NPD_WELL_BORE_CODE', 'DATEPRD']).reset_index(drop=True)

df_final.to_csv('../data/processed/Volve_Dataset_Akhir_Sempurna.csv', index=False)
print(f"[SUCCESS] {len(df_impute)} baris berhasil direstorasi dan disimpan ke file final.")